# 🛒 Olist 巴西电商销售分析
> **目标岗位：电商数据分析师**

本 Notebook 从 GMV 趋势、品类结构、用户画像三个核心维度切入，对 Olist 平台 2016-2018 年的订单数据进行系统性分析。

---
## 0. 数据加载与关联逻辑


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# 中文显示配置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei', 'Noto Sans CJK SC']
plt.rcParams['axes.unicode_minus'] = False

sns.set_style('whitegrid')

DATA_DIR = './数据集/'


In [ ]:
# ========== 加载全部 9 张表 ==========
# encoding 用 latin-1 避免 BOM/特殊字符问题

customers   = pd.read_csv(DATA_DIR + 'olist_customers_dataset.csv')
orders      = pd.read_csv(DATA_DIR + 'olist_orders_dataset.csv')
items       = pd.read_csv(DATA_DIR + 'olist_order_items_dataset.csv')
payments    = pd.read_csv(DATA_DIR + 'olist_order_payments_dataset.csv')
reviews     = pd.read_csv(DATA_DIR + 'olist_order_reviews_dataset.csv')
products    = pd.read_csv(DATA_DIR + 'olist_products_dataset.csv')
sellers     = pd.read_csv(DATA_DIR + 'olist_sellers_dataset.csv')
geolocation = pd.read_csv(DATA_DIR + 'olist_geolocation_dataset.csv')
cat_trans   = pd.read_csv(DATA_DIR + 'product_category_name_translation.csv', encoding='latin-1')

print('✅ 全部加载完成')
for name, df in [('customers',customers),('orders',orders),('items',items),
                 ('payments',payments),('reviews',reviews),('products',products),
                 ('sellers',sellers),('geolocation',geolocation),('cat_trans',cat_trans)]:
    print(f'  {name:>12s}: {df.shape[0]:>7,} 行 × {df.shape[1]} 列')

In [ ]:
# ========== 关联主表：以 orders 为中心 ==========
# 星型模型：orders 是事实表，其他是维度表

# 1. orders + items（一对多，一个订单可有多个商品）
order_items = orders.merge(items, on='order_id', how='left')

# 2. + products（商品维度）
order_items = order_items.merge(products, on='product_id', how='left')

# 3. + 品类英文翻译
order_items = order_items.merge(cat_trans, on='product_category_name', how='left')

# 4. + payments（支付信息聚合）
pay_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_methods=('payment_type', 'nunique'),
    payment_types=('payment_type', lambda x: '|'.join(sorted(x.unique())))
).reset_index()
order_items = order_items.merge(pay_agg, on='order_id', how='left')

# 5. + reviews（评分聚合）
rev_agg = reviews.groupby('order_id')['review_score'].mean().reset_index()
rev_agg.columns = ['order_id', 'avg_review_score']
order_items = order_items.merge(rev_agg, on='order_id', how='left')

# 6. + customers
order_items = order_items.merge(customers, on='customer_id', how='left')

print(f'✅ 关联完成，主表 shape: {order_items.shape}')
print(f'   列: {", ".join(order_items.columns.tolist())}')

In [ ]:
# ========== 时间字段转 datetime ==========
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date', 'shipping_limit_date']

for col in date_cols:
    if col in order_items.columns:
        order_items[col] = pd.to_datetime(order_items[col])

# 派生时间字段
order_items['purchase_year']  = order_items['order_purchase_timestamp'].dt.year
order_items['purchase_month'] = order_items['order_purchase_timestamp'].dt.to_period('M')
order_items['purchase_weekday'] = order_items['order_purchase_timestamp'].dt.day_name()
order_items['purchase_hour']   = order_items['order_purchase_timestamp'].dt.hour

print('✅ 时间解析完成')
print(f"   时间跨度: {order_items['order_purchase_timestamp'].min()} → {order_items['order_purchase_timestamp'].max()}")

---
## 1. GMV 概览 📈

> **分析目标：** 回答「这个平台生意做得怎么样」

- 月度 GMV、订单量、客单价趋势
- 订单状态分布
- 年度对比

In [ ]:
# TODO: 月度 GMV 趋势
# - 按 purchase_month 聚合 sum(price) / count(distinct order_id) / avg(price)
# - 折线图 + 柱状图双轴

pass

In [ ]:
# TODO: 订单状态分布
# - order_status 的饼图或柱状图
# - 只看 delivered 的订单做后续分析

pass

In [ ]:
# TODO: 年度对比（2016 vs 2017 vs 2018）
# - 注意 2016 年只有 4 个月数据，2018 年到 8 月
# - 注明数据口径

pass

In [ ]:
# ✅ GMV 分析结论（Markdown）
# - 峰值在哪个月？（可能是年底促销季）
# - 客单价趋势是上升还是下降？
# - 订单量增长是否健康？


---
## 2. 品类分析 🏷️

> **分析目标：** 回答「什么品类在撑这个平台」

- 品类销售额 TOP10
- 品类客单价对比
- 品类增长趋势（哪些品类在涨，哪些在跌）

In [ ]:
# TODO: 品类销售额 TOP10
# - 用 product_category_name_english 做标签
# - 水平柱状图，按 total_gmv 降序

pass

In [ ]:
# TODO: 品类客单价对比
# - scatter: x=订单量, y=客单价, size=GMV, color=品类
# - 找出「高 GMV + 高客单价 + 高销量」的明星品类

pass

In [ ]:
# TODO: 品类增长趋势
# - 按月看 TOP5 品类的 GMV 份额变化
# - 面积图或堆叠柱状图

pass

In [ ]:
# ✅ 品类分析结论
# - 长尾现象是否严重？
# - 是否有某个品类过度依赖？


---
## 3. 用户画像 👤

> **分析目标：** 回答「谁在买、怎么买」

- 地域分布（州/城市级别的订单量 & GMV）
- 下单时段（周几？几点？）
- 复购率 & 客户分层

In [ ]:
# TODO: 地域分布
# - customer_state 聚合 GMV + 订单量
# - Top 10 州的柱状图
# - （可选）用 geolocation 做城市级热力图

pass

In [ ]:
# TODO: 下单时段分析
# - 周几下单最多？（bar: Monday-Sunday）
# - 几点下单最多？（line: 0-23 小时分布）
# - 工作日 vs 周末对比

pass

In [ ]:
# TODO: 复购率
# - 定义：同一个 customer_unique_id 买了多少次
# - 一次购买 / 2-3 次 / 4+ 次 的占比
# - 复购用户 vs 新用户的 GMV 贡献

pass

In [ ]:
# ✅ 用户画像结论
# - 核心用户集中在哪几个州？
# - 复购率是否健康？
# - 是否有明显的下单高峰时段？


---
## 4. 加分维度（有余力再做）🎯

- [ ] 支付方式分析（信用卡 vs boleto vs 其他，分期比例）
- [ ] 评分 vs 销量关系（高评分品类卖得好吗？）
- [ ] 物流时效（实际送达 vs 预计送达，延迟对评分的影响）
- [ ] 卖家集中度（头部卖家贡献了多少 GMV？）


---
## 📝 总结 & 业务建议

（在所有维度做完后填写——把这些分析浓缩成3~5条可落地的业务建议）
